# 🚀 RV32I Core Floorplanning & Congestion Predictor
> **Dataset-Driven Machine Learning Framework & Interactive EDA Silicon Routing Engine for 32-Bit RISC-V Processor Cores**

---

### Project Overview
This Google Colab notebook implements an end-to-end Machine Learning pipeline for predicting routing congestion, per-metal-layer wire demand ($M_1$ to $M_{10}$), and thermal risk in **RV32I RISC-V 32-Bit Processor Core** floorplans.

In [ ]:
# Step 1: Environment Setup & Library Imports
!pip install pandas numpy scikit-learn joblib flask localtunnel matplotlib seaborn -q

import os
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Environment & Machine Learning packages ready!")

## 1. Load RV32I Floorplanning Dataset (30,500 Runs)

In [ ]:
dataset_url = "https://raw.githubusercontent.com/AzaanGIT/rv32i-congestion-predictor/main/congestion_dataset_full.csv"
df = pd.read_csv(dataset_url)

print(f"✓ Loaded Dataset: {df.shape[0]:,} floorplanning runs, {df.shape[1]} features")
display(df.head())

## 2. Exploratory Data Analysis & Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Routing Feasibility Countplot
sns.countplot(data=df, x='success', palette='Set2', ax=axes[0])
axes[0].set_title('Routing Feasibility (Success vs Failure)')
axes[0].set_xticklabels(['DRC Failure (0)', 'Routable Clean (1)'])

# 2. Per-Layer Wire Demand Boxplot
demand_cols = [f'metal{i}_demand' for i in range(1, 11)]
df[demand_cols].boxplot(ax=axes[1], rot=45)
axes[1].set_title('Wire Demand Across Metal Layers (M1 - M10)')
axes[1].set_ylabel('Tracks')

plt.tight_layout()
plt.show()

## 3. Machine Learning Model Training & Comparative Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, r2_score, mean_absolute_error

FEATURES = ["utilization", "aspect_ratio", "core_margin", "density", "layer_adj"]
LAYERS = [f"metal{i}" for i in range(1, 11)]
DEMAND_TARGETS = [f"{l}_demand" for l in LAYERS]

# --- 1. CLASSIFIER COMPARISON (ROUTING FEASIBILITY) ---
X = df[FEATURES]
y_cls = df["success"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

classifiers = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

print("=== CLASSIFIER ACCURACY COMPARISON ===")
for name, model in classifiers.items():
    model.fit(Xtr, ytr)
    acc = accuracy_score(yte, model.predict(Xte))
    print(f"  {name:22s} Accuracy: {acc*100:.2f}%")

clf = classifiers["Random Forest"]

# --- 2. MULTI-OUTPUT REGRESSOR (WIRE DEMAND PREDICTION) ---
routed = df[df["success"] == True].copy()
Xr = routed[FEATURES]
regressors = {}
metrics = {}

for target in DEMAND_TARGETS + ["Total_demand"]:
    yr = routed[target]
    Xrtr, Xrte, yrtr, yrte = train_test_split(Xr, yr, test_size=0.2, random_state=42)
    reg = RandomForestRegressor(n_estimators=80, max_depth=14, min_samples_leaf=3, random_state=42, n_jobs=-1)
    reg.fit(Xrtr, yrtr)
    pred = reg.predict(Xrte)
    r2 = r2_score(yrte, pred)
    mae = mean_absolute_error(yrte, pred)
    metrics[target] = {"r2": r2, "mae": mae}
    regressors[target] = reg

avg_r2 = np.mean([v["r2"] for v in metrics.values()])
print(f"\n=== REGRESSOR PERFORMANCE ===")
print(f"  Average R² Score (M1-M10): {avg_r2:.4f}")

## 4. Export Machine Learning Artifacts

In [ ]:
os.makedirs("model", exist_ok=True)
joblib.dump(regressors, "model/congestion_regressors.joblib")
joblib.dump(clf, "model/success_classifier.joblib")

meta = {
    "classifier_accuracy": float(accuracy_score(yte, clf.predict(Xte))),
    "regressor_metrics": metrics,
    "default_resource": {f"metal{i}": float(routed[f"metal{i}_resource"].median()) for i in range(1, 11)}
}

with open("model/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("✓ Model artifacts saved to model/ directory!")

## 5. Launch Web Application & Generate Public Colab URL

In [ ]:
import subprocess

if not os.path.exists("app.py"):
    !git clone https://github.com/AzaanGIT/rv32i-congestion-predictor.git repo
    %cd repo/congestion_app

# Start Flask Server in Background
subprocess.Popen(["python3", "app.py"])
time.sleep(3)

print("🌐 Website Server Online at http://localhost:5000/")
print("Generating public shareable URL via localtunnel...")

!npx localtunnel --port 5000